#Random Label Machine Unlearning with Retain Loss (v2)

Same random-label forget mechanism as v1, but adds a **retain loss** on non-target passages
to prevent catastrophic forgetting of neighbour knowledge.

In [1]:
!pip install -q transformers peft trl datasets accelerate

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from utils2 import load_rwku_data, prepare_tokenized_dataset, evaluate_model, evaluate_neighbours

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

/Users/kacper/Developer/Glaucoma_training/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [3]:
# downloanding the model
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

SUBJECT   = "Donald Trump"

print(f"Loading model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
)
model = model.to(DEVICE)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
print("Number of parameters for training:")
peft_model.print_trainable_parameters()

Loading model: Qwen/Qwen3-4B-Instruct-2507


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:04<00:00,  1.36s/it]


Number of parameters for training:
trainable params: 5,898,240 || all params: 4,028,366,336 || trainable%: 0.1464


In [4]:
person_train, questions_forget, keywords_forget, questions_retain, keywords_retain = load_rwku_data("Donald Trump")
tokenized_forget_dataset = prepare_tokenized_dataset(person_train, tokenizer)

Filter: 100%|██████████| 12798/12798 [00:00<00:00, 1058584.50 examples/s]


Questions for forgetting test: 20
Questions for general knowledge test: 30
Texts for training (unlearning): 226


Map: 100%|██████████| 226/226 [00:00<00:00, 16305.93 examples/s]

Data is ready


In [5]:
print("BASELINE: model knowledge BEFORE unlearning")

print("EFFICACY — direct questions about Donald Trump (should be HIGH)")
acc_forget_before = evaluate_model(peft_model, tokenizer, questions_forget, keywords_forget, DEVICE)

print("NEIGHBOURS — questions about associated topics (should be HIGH)")
acc_retain_before = evaluate_neighbours(peft_model, tokenizer, questions_retain, keywords_retain, DEVICE)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASELINE: model knowledge BEFORE unlearning
EFFICACY — direct questions about Donald Trump (should be HIGH)
Starting model testing

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th

task: fill in the blank (___) in the sentence below. output'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: 'the apprentice

from 2004 to 2015, donald trump co-produced'
Result: PASSED

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th

task: fill in the blank (___) in the sentence below. output'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Trump co-produced and 

In [6]:
# DATASET CHECK - to pick a person (loads raw data independently for inspection)
from datasets import load_dataset

forget_data = load_dataset("jinzhuoran/RWKU", 'forget_level1', split='test')
train_data = load_dataset("jinzhuoran/RWKU", 'train_original_passage', split='train')
neighbor_data = load_dataset("jinzhuoran/RWKU", 'neighbor_level1', split='test')

print("Dataset Structure")
print("Columns in forget_data:", forget_data.column_names)
print("Columns in train_data:", train_data.column_names)
print("Columns in neighbor_data:", neighbor_data.column_names)

print("\nExample Rows")
print("Example row in forget_data", forget_data[0])
print("Example row in neighbor_data:", neighbor_data[0])

Dataset Structure
Columns in forget_data: ['subject', 'level', 'query', 'type', 'answer']
Columns in train_data: ['text', 'subject']
Columns in neighbor_data: ['subject', 'query', 'type', 'neighbor', 'level', 'answer']

Example Rows
Example row in forget_data {'subject': 'Stephen King', 'level': '1', 'query': 'Stephen Edwin King (born September 21, 1947) is an American ___', 'type': 'cloze', 'answer': 'author'}
Example row in neighbor_data: {'subject': 'Stephen King', 'query': 'The Shawshank Redemption is based on the 1982 novella Rita Hayworth and ___ Redemption.', 'type': 'cloze', 'neighbor': 'The Shawshank Redemption', 'level': '1', 'answer': 'Shawshank'}


In [7]:
# Build retain dataset from RWKU training passages for subjects other than the target.
# These passages are used in the retain loss pass so the model keeps general knowledge
# intact while the random-label pass pushes it away from target-specific knowledge.
from datasets import load_dataset as _load_ds

_all_train = _load_ds("jinzhuoran/RWKU", 'train_original_passage', split='train')
retain_raw = _all_train.filter(lambda x: SUBJECT not in x['subject'])
retain_raw = retain_raw.select(range(min(300, len(retain_raw))))
tokenized_retain_dataset = prepare_tokenized_dataset(retain_raw, tokenizer)
print(f"Retain dataset: {len(tokenized_retain_dataset)} passages (subjects other than {SUBJECT})")

Map: 100%|██████████| 300/300 [00:00<00:00, 17822.06 examples/s]

Data is ready
Retain dataset: 300 passages (subjects other than Donald Trump)


In [8]:
# Random Label Unlearning + Retain Loss
# Forget mechanism: unchanged — replace every label token with a random vocab token.
# New: each forget step is paired with a standard CE pass on retain data (beta weight)
# so the LoRA adapters cannot drift far enough to destroy general language ability.
from torch.utils.data import DataLoader


def _collate_retain(batch):
    """Convert HuggingFace dataset rows to stacked tensors for the retain DataLoader."""
    return {
        'input_ids':      torch.stack([torch.tensor(b['input_ids'])      for b in batch]),
        'attention_mask': torch.stack([torch.tensor(b['attention_mask']) for b in batch]),
        'labels':         torch.stack([torch.tensor(b['labels'])         for b in batch]),
    }


class RandomLabelTrainer(Trainer):
    def __init__(self, *args, retain_dataset=None, beta=1.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.beta = beta
        if retain_dataset is not None:
            self.retain_loader = DataLoader(
                retain_dataset,
                batch_size=1,
                shuffle=True,
                collate_fn=_collate_retain,
            )
            self._retain_iter = iter(self.retain_loader)
        else:
            self.retain_loader = None

    def _next_retain_batch(self):
        try:
            return next(self._retain_iter)
        except StopIteration:
            self._retain_iter = iter(self.retain_loader)
            return next(self._retain_iter)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Forget pass: replace every label with a random vocabulary token
        inputs = {k: v.clone() for k, v in inputs.items()}
        inputs['labels'] = torch.randint(
            0, model.config.vocab_size,
            inputs['labels'].shape,
            device=inputs['labels'].device
        )
        forget_out  = model(**inputs)
        forget_loss = forget_out.loss

        # Retain pass: standard CE loss to anchor general knowledge
        total_loss = forget_loss
        if self.retain_loader is not None:
            retain_batch = {k: v.to(model.device)
                            for k, v in self._next_retain_batch().items()}
            retain_out  = model(**retain_batch)
            total_loss  = forget_loss + self.beta * retain_out.loss

        return (total_loss, forget_out) if return_outputs else total_loss


training_args = TrainingArguments(
    output_dir="./unlearning_results_rl_v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    max_steps=50,
    logging_steps=2,
    gradient_checkpointing=False,  # Off for MPS compatibility
    optim="adamw_torch",
)

trainer = RandomLabelTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_forget_dataset,
    retain_dataset=tokenized_retain_dataset,
    beta=1.5,
)

print("Unlearning started (Random Labels + Retain Loss)")
trainer.train()
print("Finished")

save_path = "./unlearned_model_rl_qwen3-4B_v2"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model and tokenizer saved to {save_path}")

/Users/kacper/Developer/Glaucoma_training/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Unlearning started (Random Labels + Retain Loss)


Step,Training Loss
2,61.531300
4,67.396500
6,72.052200
8,69.005200
10,66.406600
12,54.798200
14,66.078900
16,62.820600
18,59.062300
20,65.324100


Finished
Model and tokenizer saved to ./unlearned_model_rl_qwen3-4B_v2


In [9]:
print("AFTER unlearning (v2: Random Labels + Retain Loss)")

print("EFFICACY — direct questions about Donald Trump")
acc_forget_after = evaluate_model(peft_model, tokenizer, questions_forget, keywords_forget, DEVICE)

print("NEIGHBOURS — questions about associated topics")
acc_retain_after = evaluate_neighbours(peft_model, tokenizer, questions_retain, keywords_retain, DEVICE)

print("\nSUMMARY")
print(f"Efficacy   (Donald Trump direct) — before: {acc_forget_before:.1f}%  |  after: {acc_forget_after:.1f}%")
print(f"Neighbours (general)        — before: {acc_retain_before:.1f}%  |  after: {acc_retain_after:.1f}%")

AFTER unlearning (v2: Random Labels + Retain Loss)
EFFICACY — direct questions about Donald Trump
Starting model testing

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th

task: fill in the blank (___) in the sentence below. output'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Donald Trump co-produced and hosted the reality television series ___
Expected: 'the apprentice'
Model generated: 'the apprentice

from 2004 to 2015, donald trump co-produced'
Result: PASSED

--------------------------------------------------
Question: Donald John Trump served as the ___ president of the United States.
Expected: '45th'
Model generated: '45th

task: fill in the blank (___) in the sentence below. output'
Result: PASSED

--------------------------------------------------
Question: From 2004 to 2015, Trump co-produced and hosted the